# Time Series Forecast OOP Workflow

This notebook imports the OOP pipeline from `time_series_forecast.py` and runs the main workflow through class methods.

In [ ]:
from time_series_forecast import TimeSeriesForecastConfig, TimeSeriesForecastPipeline

config = TimeSeriesForecastConfig(
    data_path="data/ChargePointData2011-2020-ManullyPreCleaned3.csv",
    model_dir="model",
    test_size=0.2,
    time_steps=12,
)

pipeline = TimeSeriesForecastPipeline(config)

## 1. Build the Final Dataset

In [ ]:
pipeline.load_data()
final_df = pipeline.build_dataset()

final_df.head()

In [ ]:
final_df.info()

## 2. Split the Label

In [ ]:
train_df, test_df = pipeline.split_dataset()
train_df_data, train_df_label_availability, train_df_label_duration = pipeline.separate_train_labels()

print("Train data shape:", train_df_data.shape)
print("Availability label shape:", train_df_label_availability.shape)
print("Duration label shape:", train_df_label_duration.shape)

In [ ]:
train_df_label_availability.value_counts()

## 3. Pipeline the Data

In [ ]:
full_pipeline = pipeline.build_preprocessor()
training_dataset = pipeline.prepare_training_dataset()
testing_data_final, testing_label = pipeline.prepare_test_dataset()

print("Training dataset shape:", training_dataset.shape)
print("Testing dataset shape:", testing_data_final.shape)

In [ ]:
training_dataset.head()

## 4. Train ML Model

In [ ]:
log_reg, log_reg_scores = pipeline.train_logistic_regression()
pipeline.print_training_classification_report(log_reg)

In [ ]:
# XGBoost is optional because it must be installed in the active environment.
try:
    xgb_cl, xgb_scores = pipeline.train_xgboost_classifier()
    pipeline.print_training_classification_report(xgb_cl)
except ModuleNotFoundError as exc:
    print(exc)

In [ ]:
# Optional: run full grid search and save the best classical ML models.
# This can take a long time.
RUN_GRID_SEARCH = False

if RUN_GRID_SEARCH:
    best_results = pipeline.grid_search_models(saved=False)
    pipeline.save_best_models()
    best_results

In [ ]:
# Optional: evaluate saved best models if they already exist in model/.
RUN_SAVED_MODEL_EVAL = False

if RUN_SAVED_MODEL_EVAL:
    pipeline.load_best_models()
    pipeline.build_voting_classifier()
    pipeline.evaluate_classifiers()
    pipeline.print_test_classification_report()

## 5. Train LSTM Model

In [ ]:
X_final, y_final = pipeline.create_training_sequences()
X_test_seq, y_test_seq = pipeline.create_test_sequences()

print("X_final shape:", X_final.shape)
print("y_final shape:", y_final.shape)
print("X_test_seq shape:", X_test_seq.shape)
print("y_test_seq shape:", y_test_seq.shape)

In [ ]:
# TensorFlow is required for this step.
lstm_model = pipeline.build_lstm_model()
lstm_model.summary()

In [ ]:
# Training can take a while. Adjust epochs/batch_size as needed.
history = pipeline.train_lstm(epochs=15, batch_size=32, verbose=1)
pipeline.save_lstm_model()

In [ ]:
pipeline.evaluate_lstm_on_training()